# 06 — Drift Monitoring and Model Governance

Reviews the drift report, fairness analysis, and model governance artifacts produced by the pipeline.

In [ ]:
import sys
sys.path.insert(0, "..")

import json
import pandas as pd
import matplotlib.pyplot as plt

## Feature drift (PSI)

In [ ]:
with open("../reports/drift_report.json") as f:
    drift = json.load(f)

feature_drift = drift["feature_drift"]
psi_df = pd.DataFrame({
    "feature": list(feature_drift.keys()),
    "psi": [v["psi"] for v in feature_drift.values()],
    "drift_flag": [v["drift_flag"] for v in feature_drift.values()],
}).sort_values("psi", ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ["salmon" if f else "steelblue" for f in psi_df["drift_flag"]]
ax.barh(psi_df["feature"], psi_df["psi"], color=colors)
ax.axvline(x=0.2, color="red", linestyle="--", label="Drift threshold (0.2)")
ax.set_xlabel("PSI")
ax.set_title("Feature drift: Population Stability Index")
ax.legend()
plt.tight_layout()
plt.show()

## Score drift

In [ ]:
score_drift = drift["score_drift"]
for score, info in score_drift.items():
    flag = " *** DRIFT ***" if info["drift_flag"] else ""
    print(f"{score:35s}  PSI={info['psi']:.6f}{flag}")

## Alert volume comparison

In [ ]:
avd = drift["alert_volume_drift"]
print("Train action distribution:", avd["train_action_distribution"])
print("Test action distribution: ", avd["test_action_distribution"])

## Test set performance

In [ ]:
perf = drift["test_set_performance"]
for k, v in perf.items():
    print(f"{k:15s}: {v:.4f}")

## Fairness report

In [ ]:
try:
    with open("../reports/fairness_report.json") as f:
        fairness = json.load(f)
    for segment, values in fairness.items():
        print(f"\n=== {segment} ===")
        for group, metrics in values.items():
            print(f"  {group:15s}  FPR={metrics['false_positive_rate']:.4f}  (FP={metrics['false_positive_count']}, N={metrics['total_negatives']})")
except FileNotFoundError:
    print("Fairness report not found. Run train_pipeline.py first.")

## Key findings

- PSI-based drift monitoring flags features whose distributions have shifted significantly between training and test periods.
- Score drift tracking ensures the model's output behaviour remains stable.
- Fairness monitoring checks for disparate false-positive rates across protected segments.
- These governance artifacts support regulatory and audit requirements.